# 1. Project Introduction

Parkinson’s disease (PD) is a progressive neurological disorder that affects movement and speech production. Early detection can support clinical decision-making and patient care. 

In this project, machine learning techniques are applied to biomedical voice measurements in order to classify individuals as healthy or affected by Parkinson’s disease. The dataset contains multiple vocal and nonlinear dynamical features extracted from sustained phonations.

The objective of this project is to:
- Explore and preprocess biomedical voice data
- Train a Support Vector Machine (SVM) classifier
- Compare model performance with and without feature scaling
- Evaluate classification performance using multiple metrics
- Generate predictions on unseen data

This project also demonstrates the importance of preprocessing pipelines and reproducible machine learning workflows using Scikit-learn.

# 2. Import Libraries

The following libraries are used throughout the project:

- **NumPy** for numerical operations
- **Pandas** for data manipulation and analysis
- **Scikit-learn** for preprocessing, model training, evaluation, and cross-validation

The project uses:
- `StandardScaler` for feature scaling
- `Pipeline` for combining preprocessing and modeling
- `SVC` (Support Vector Classifier) for supervised classification

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 3. Load Dataset

The dataset is loaded using Pandas from the Kaggle input directory.

The dataset contains:
- 195 voice recordings
- 31 individuals
- 23 individuals diagnosed with Parkinson’s disease

The target variable:
- `status = 1` → Parkinson’s disease
- `status = 0` → Healthy individual

Each row corresponds to a voice recording and includes biomedical voice measurements such as:
- Fundamental vocal frequencies
- Jitter measurements
- Shimmer measurements
- Noise-to-harmonics ratios
- Nonlinear dynamical complexity measures

In [2]:
parkdf = pd.read_csv(
    "/kaggle/input/datasets/vikasukani/parkinsons-disease-data-set/parkinsons.data"
)

# 4. Exploratory Data Analysis

Basic exploratory analysis is performed to better understand the dataset structure and quality.

The following checks are conducted:
- Dataset information and feature types using `.info()`
- Missing values using `.isna().sum()`
- Class distribution using `.value_counts()`

This step helps identify:
- Potential missing data
- Class imbalance
- Data types and dimensionality

In [18]:
# Basic checks
print(parkdf.info())
print(parkdf.isna().sum())
print(parkdf["status"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              195 non-null    object 
 1   MDVP:Fo(Hz)       195 non-null    float64
 2   MDVP:Fhi(Hz)      195 non-null    float64
 3   MDVP:Flo(Hz)      195 non-null    float64
 4   MDVP:Jitter(%)    195 non-null    float64
 5   MDVP:Jitter(Abs)  195 non-null    float64
 6   MDVP:RAP          195 non-null    float64
 7   MDVP:PPQ          195 non-null    float64
 8   Jitter:DDP        195 non-null    float64
 9   MDVP:Shimmer      195 non-null    float64
 10  MDVP:Shimmer(dB)  195 non-null    float64
 11  Shimmer:APQ3      195 non-null    float64
 12  Shimmer:APQ5      195 non-null    float64
 13  MDVP:APQ          195 non-null    float64
 14  Shimmer:DDA       195 non-null    float64
 15  NHR               195 non-null    float64
 16  HNR               195 non-null    float64
 1

# 5. Data Preprocessing and Train/Test Split

Before training the model, the dataset is prepared for machine learning.

The `name` column is removed because it only contains recording identifiers and does not provide predictive information for the classification task.

The dataset is then separated into:
- `X` → input features
- `y` → target labels

Next, the data is divided into:
- Training set (80%)
- Testing set (20%)

A stratified split is used to preserve the proportion of healthy and Parkinson’s cases in both subsets.

The parameter: stratify=y

ensures that class distribution remains balanced between training and testing datasets, improving the reliability of model evaluation.

In [3]:
# Features and target
X = parkdf.drop(columns=["name", "status"])
y = parkdf["status"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2, stratify=y
)

# 6. Models creation

Two modeling approaches are explored:

## Model 1 — SVM Without Feature Scaling

A linear Support Vector Machine (SVM) classifier is first trained directly on the raw input features without preprocessing. This model serves as a baseline for comparison.

## Model 2 — SVM With Feature Scaling

A second SVM classifier is trained using feature scaling. Feature scaling is particularly important for Support Vector Machines because SVMs are sensitive to differences in feature magnitudes.

A Scikit-learn `Pipeline` is used to combine:
1. `StandardScaler`
2. Linear `SVC`

This automates preprocessing and modeling in a reproducible workflow while preventing data leakage during scaling.

In [8]:
# Model 1: SVM without scaling

model_without_scaling = SVC(kernel="linear")

model_without_scaling.fit(X_train, y_train)

# Model 2: SVM with scaling using Pipeline

model_with_scaling = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="linear"))
])

model_with_scaling.fit(X_train, y_train)

Accuracy score on training data without scaling: 0.8653846153846154
Accuracy score on testing data without scaling: 0.8461538461538461
Accuracy score on training data with scaling: 0.8974358974358975
Accuracy score on testing data with scaling: 0.8974358974358975
Mean cross-validation accuracy with scaling: 0.7948717948717949
Mean cross-validation accuracy without scaling: 0.8256410256410256


# 7. Model Evaluation

The models are evaluated using several classification metrics:

- Training accuracy
- Testing accuracy
- Confusion matrix
- Classification report

## Accuracy and Cross-Validation Accuracy

Before training the final model, cross-validation is performed using:

cross_val_score()

in Model 1 and 2. Accuracy after the cross validation is compared.

Although feature scaling is commonly recommended for Support Vector Machines, the non-scaled linear SVM achieved slightly higher cross-validation accuracy on this dataset. These results highlight that preprocessing choices should be evaluated empirically rather than assumed to systematically improve model performance.

Comparing models with and without scaling helps demonstrate the impact of preprocessing on SVM performance.

## Sensitivity and Precision of the models

The confusion matrix provides insight into:
- True positives
- True negatives
- False positives
- False negatives

The classification report includes:
- Precision
- Recall
- F1-score

The scaled SVM model achieved better performance on the held-out test set, particularly for the classification of healthy individuals, while cross-validation results showed slightly higher average accuracy for the non-scaled model.

These differences suggest that model performance may vary across dataset splits due to the relatively small sample size. Overall, feature scaling improved classification balance and produced stronger evaluation metrics on the final test set.

In [9]:
# Models evaluation 

train_pred_without_scaling = model_without_scaling.predict(X_train)
test_pred_without_scaling = model_without_scaling.predict(X_test)

train_pred_with_scaling = model_with_scaling.predict(X_train)
test_pred_with_scaling = model_with_scaling.predict(X_test)

print("Accuracy score on training data without scaling:",
      accuracy_score(y_train, train_pred_without_scaling)) # accuracy for 1 train-test-split

print("Accuracy score on testing data without scaling:",
      accuracy_score(y_test, test_pred_without_scaling))  # accuracy for 1 train-test split

print("Accuracy score on training data with scaling:",
      accuracy_score(y_train, train_pred_with_scaling)) # accuracy for 1 train-test-split

print("Accuracy score on testing data with scaling:",
      accuracy_score(y_test, test_pred_with_scaling))  # accuracy for 1 train-test split

# Cross-validation of Model 1 and Model 2

cv_no_scaling = cross_val_score(
    model_without_scaling,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(
    "Mean cross-validation accuracy without scaling:",
    cv_no_scaling.mean()
)

cv_with_scaling = cross_val_score(
    model_with_scaling,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(
    "Mean cross-validation accuracy with scaling:",
    cv_with_scaling.mean()
)

print("Confusion matrix without scaling:")
print(confusion_matrix(y_test, test_pred_without_scaling))

print("Classification report without scaling:")
print(classification_report(y_test, test_pred_without_scaling))

print("Confusion matrix with scaling:")
print(confusion_matrix(y_test, test_pred_with_scaling))

print("Classification report with scaling:")
print(classification_report(y_test, test_pred_with_scaling))


Accuracy score on training data without scaling: 0.8653846153846154
Accuracy score on testing data without scaling: 0.8461538461538461
Accuracy score on training data with scaling: 0.8974358974358975
Accuracy score on testing data with scaling: 0.8974358974358975
Mean cross-validation accuracy without scaling: 0.8256410256410256
Mean cross-validation accuracy with scaling: 0.7948717948717949
Confusion matrix without scaling:
[[ 6  4]
 [ 2 27]]
Classification report without scaling:
              precision    recall  f1-score   support

           0       0.75      0.60      0.67        10
           1       0.87      0.93      0.90        29

    accuracy                           0.85        39
   macro avg       0.81      0.77      0.78        39
weighted avg       0.84      0.85      0.84        39

Confusion matrix with scaling:
[[ 8  2]
 [ 2 27]]
Classification report with scaling:
              precision    recall  f1-score   support

           0       0.80      0.80      0.80  

# 8. Predictions

After training, the final model is used to generate predictions on unseen input data.

An example voice recording is provided as input to the trained classifier. The model predicts whether the recording corresponds to:
- A healthy individual
- A person with Parkinson’s disease

This demonstrates how trained machine learning models can be applied to new observations after preprocessing and training.

In [10]:
# Example prediction on new input data

input_data = (
    119.99200, 157.30200, 74.99700, 0.00784, 0.00007,
    0.00370, 0.00554, 0.01109, 0.04374, 0.42600,
    0.02182, 0.03130, 0.02971, 0.06545, 0.02211,
    21.03300, 0.414783, 0.815285, -4.813031,
    0.266482, 2.301442, 0.284654
)

input_df = pd.DataFrame([input_data], columns=X.columns)

prediction = model_with_scaling.predict(input_df)

if prediction[0] == 0:
    print("The person is predicted not to have Parkinson's disease.")
else:
    print("The person is predicted to have Parkinson's disease.")




The person is predicted to have Parkinson's disease.


# 9. Conclusion

This project demonstrates a complete supervised machine learning workflow applied to biomedical voice data for Parkinson’s disease classification.

Key steps included:
- Exploratory data analysis
- Data preprocessing
- Feature scaling
- SVM classification
- Model evaluation
- Prediction on new data

The results highlight the importance of preprocessing, particularly feature scaling, when working with Support Vector Machines.

Beyond predictive performance, this project emphasizes reproducible machine learning practices using Scikit-learn pipelines and structured workflows.

Future improvements could include:
- Testing additional classification models
- Hyperparameter optimization
- Feature importance analysis
- Advanced visualization techniques
- Deployment of the model as a simple web application